# 05 — 变分推断与 VAE

第 04 章直接 MLE，第 04.5 章研究边缘积分的计算困难。现在仍使用同一个生成模型：

$$p(x)=N(0,1),\qquad p_\theta(y|x)=N(f_\theta(x),\sigma^2I).$$

原目标为 $\max_\theta\sum_i\log p_\theta(y_i)$。由于边缘积分困难，引入近似后验 $q_\phi(x|y)=N(m_\phi(y),s_\phi(y)^2)$。它不是模型的真实后验。

$$\log p_\theta(y)=\mathcal L(\theta,\phi;y)+D_{KL}(q_\phi(x|y)\|p_\theta(x|y)).$$

$$\mathcal L=\mathbb E_{x\sim q_\phi(x|y)}[\log p_\theta(y|x)+\log p(x)-\log q_\phi(x|y)].$$

本章直接计算这三项。固定当前生成模型、只优化 q 是变分推断；同时优化生成模型和 q 是变分学习。第一阶段用已训练的第 04 章生成模型单独展示推断，第二阶段从随机初始化联合训练。第一阶段不是第二阶段的必要前置训练。传统变分推断可以对每个 y 单独优化均值、标准差；这里使用共享神经网络，根据 y 输出这些参数，即摊销变分推断。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, multivariate_normal
from utils import generate_observations, compare, plot_history

# 共享数据，只观测 y；不读取真实生成类别。
rng = np.random.default_rng(42)
y = generate_observations(rng)

## 1. 复用第 04 章生成模型与数值积分
网格积分仍用于低维验证和准备一个固定生成模型，但不再作为 VAE 联合训练的损失。新增的编码器输出近似后验的均值和标准差，而非生成模型的参数。

In [ ]:
import torch
from torch import nn
from pathlib import Path
torch.set_num_threads(1)
torch.manual_seed(42)
Y = torch.tensor(y, dtype=torch.float32)
sigma = .45
# 与第 04 章完全相同的生成网络。
def make_decoder():
    return nn.Sequential(nn.Linear(1,32), nn.Tanh(), nn.Linear(32,32), nn.Tanh(), nn.Linear(32,2))
decoder = make_decoder()

# grid 的每一行是一个标量隐变量，形状 [G, 1]。
def log_joint(obs, grid):
    # 每个观测 y 对应 G 个候选 x；返回 [N, G]。
    prior = torch.distributions.Normal(0., 1.)
    log_prior = prior.log_prob(grid[:, 0])
    means = decoder(grid)
    # 每个候选 x 定义一个二维高斯条件分布。
    # Independent 将两个独立坐标的 log 密度相加，
    # 因此 log_prob 返回每个二维向量的联合 log 密度。
    conditional = torch.distributions.Independent(
        torch.distributions.Normal(means, sigma), 1
    )
    rows = []
    for y_i in obs:
        # 对同一个 y_i，在 G 个条件分布上求值。
        repeated_y = y_i.repeat(len(grid), 1)
        log_conditional = conditional.log_prob(repeated_y)
        log_joint_values = log_prior + log_conditional
        rows.append(log_joint_values)
    return torch.stack(rows)

def grid_log_marginal(obs, grid):
    log_joint_values = log_joint(obs, grid)
    dx = grid[1, 0] - grid[0, 0]
    values = []
    for row in log_joint_values:
        # p(y) ≈ sum_j p(x_j,y) * dx。
        # logsumexp 避免先 exp 后求和时的浮点下溢。
        log_marginal = torch.logsumexp(row, dim=0) + torch.log(dx)
        values.append(log_marginal)
    return torch.stack(values)

# 优先沿用上一章的参数；没有文件才重新训练相同模型。
if Path('decoder_grid.pt').exists():
    decoder.load_state_dict(torch.load('decoder_grid.pt', weights_only=True))
else:
    g = torch.linspace(-6,6,241).reshape(-1,1)
    opt = torch.optim.Adam(decoder.parameters(), lr=.01)
    for _ in range(350):
        obs = Y[torch.randint(len(Y), (128,))]
        loss = -grid_log_marginal(obs,g).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()

# 新增编码器：输入二维 y，输出一维隐变量的均值和 log 标准差。
encoder = nn.Sequential(nn.Linear(2,32), nn.Tanh(), nn.Linear(32,2))
def posterior_params(obs):
    output = encoder(obs)
    # 第 0 列为均值，第 1 列为 log 标准差。
    # exp 保证标准差为正；clamp 限制极端尺度，避免数值溢出。
    # 这是实现上的范围限制，不是变分推断的数学要求。
    return output[:,0], output[:,1].clamp(-5,2)



## 2. 直接计算 ELBO

对单条观测 y，先构造 q，再采样 M 次。每次计算三个对数密度，最后求平均。rsample 与 sample 的分布相同，但前者使用 $x=m+s\epsilon,\ \epsilon\sim N(0,1)$，使 x 对 m、s 可微。因为要训练近似后验网络，不能切断这条梯度路径。对生成网络参数的梯度不依赖这一技巧。

先阅读 elbo_single，再阅读用于训练的批量版本 elbo。批量版没有改变目标，只将 N 条观测同时计算。Independent 将二维条件分布的两个独立坐标作为一个事件，其 log_prob 等于两坐标对数密度之和。

In [ ]:
def elbo_single(y_i, n_samples=8):
    # q_phi(x|y_i)：先确定分布参数，再定义分布。
    m, log_s = posterior_params(y_i.reshape(1, 2))
    # posterior_params 接收一批观测，这里只有一条，取第 0 项。
    q = torch.distributions.Normal(m[0], log_s[0].exp())
    prior = torch.distributions.Normal(0., 1.)
    total = 0.0
    for j in range(n_samples):
        # rsample 等价于 x=m+s*epsilon，保留对 m、s 的梯度。
        x = q.rsample()
        mean = decoder(x.reshape(1))
        conditional = torch.distributions.MultivariateNormal(
            mean, sigma**2 * torch.eye(2)
        )
        log_conditional = conditional.log_prob(y_i)
        log_prior = prior.log_prob(x)
        log_q = q.log_prob(x)
        total = total + log_conditional + log_prior - log_q
    return total / n_samples

def elbo(obs, n_samples=8):
    # 与 elbo_single 完全相同的公式；同时处理 N 条观测。
    # m、s、x、log_prior、log_q、log_conditional 都是 [N]。
    m, log_s = posterior_params(obs)
    q = torch.distributions.Normal(m, log_s.exp())
    prior = torch.distributions.Normal(0., 1.)
    total = torch.zeros(len(obs), dtype=obs.dtype, device=obs.device)
    for j in range(n_samples):
        x = q.rsample()
        mean = decoder(x.reshape(-1, 1))
        # mean 为 [N, 2]；Independent 将二维观测作为一个事件。
        conditional = torch.distributions.Independent(
            torch.distributions.Normal(mean, sigma), 1
        )
        log_conditional = conditional.log_prob(obs)
        log_prior = prior.log_prob(x)
        log_q = q.log_prob(x)
        total = total + log_conditional + log_prior - log_q
    # 第一次平均是对每条观测的 M 个隐变量样本估计期望。
    elbo_per_observation = total / n_samples
    # 第二次平均是对 N 条观测构造训练目标。
    return elbo_per_observation.mean()


## 3. 固定生成模型：只验证变分推断
这里的生成参数来自第 04 章的 MLE，不是最终联合训练的随机初始参数。冻结参数后仍需保留 decoder 对输入 x 的梯度，因此不能用 torch.no_grad 包裹 ELBO 中的 decoder。

In [ ]:
# 第一阶段：固定生成模型，只学习近似后验。
for p in decoder.parameters():
    p.requires_grad_(False)
opt = torch.optim.Adam(encoder.parameters(),lr=.005)
for _ in range(500):
    obs = Y[torch.randint(len(Y),(128,))]
    loss = -elbo(obs)
    opt.zero_grad()
    loss.backward()
    opt.step()

In [ ]:
# 沿用第 04 章的网格后验，作为低维参考；不参与训练。
def compare_posterior(index):
    with torch.no_grad():
        obs = Y[index:index+1]
        g = torch.linspace(-6,6,1001).reshape(-1,1)
        lj = log_joint(obs,g)[0]
        log_z = grid_log_marginal(obs,g)[0]
        m, log_s = posterior_params(obs)
        q = torch.distributions.Normal(m[0],log_s[0].exp())
        model_values = []
        q_values = []
        for k in range(len(g)):
            model_values.append(torch.exp(lj[k]-log_z).item())
            q_values.append(q.log_prob(g[k,0]).exp().item())
        plt.plot(g[:,0],model_values,label='model posterior (grid)')
        plt.plot(g[:,0],q_values,label='q')
        plt.legend()
        plt.show()
        print('log evidence:',log_z.item(),'MC ELBO:',elbo(obs,1000).item())
compare_posterior(0)
compare_posterior(12)

固定生成模型时，最大化 ELBO 等价于最小化 q 与真实后验之间的 KL 散度。高斯 q 不一定能表示真实后验的多个峰；编码器也不一定能对每个样本达到最优。蒙特卡洛估计的 ELBO 可能因采样误差偶尔超过数值似然。

In [ ]:
# 第二阶段：重新初始化两个网络，联合最大化 ELBO。
# 不再调用上一章的网格积分计算训练损失。
decoder=make_decoder()
encoder=nn.Sequential(nn.Linear(2,32),nn.Tanh(),nn.Linear(32,2))
opt=torch.optim.Adam(list(decoder.parameters())+list(encoder.parameters()),lr=.005)
history=[]
for _ in range(1000):
    obs=Y[torch.randint(len(Y),(128,))]
    loss = -elbo(obs)
    opt.zero_grad()
    loss.backward()
    opt.step()
    history.append(-loss.item())
plot_history(history, 'MC ELBO')
compare_posterior(0)
# 生成数据仍然只使用先验和解码器，编码器不参与。
with torch.no_grad():
    samples = []
    for i in range(len(y)):
        x = torch.randn(1)
        conditional = torch.distributions.Normal(decoder(x), sigma)
        samples.append(conditional.sample().numpy())
    compare(y, np.array(samples))

改变隐变量维度将需要同时调整先验、编码器和解码器；训练时的采样仍可扩展，不需要指数增长的网格。VAE 的生成质量受模型、后验近似和优化影响，不保证拟合优于直接 MLE。

## 可选：解析 KL 对照

利用期望线性性质，ELBO 也等于 $\mathbb E_q[\log p_\theta(y|x)]-D_{KL}(q_\phi(x|y)\|p(x))$。当前两个高斯的 KL 恰好有解析解 $\frac12(m^2+s^2-1-2\log s)$。这只是计算简化，不是新的训练目标。解析计算可以减少采样波动；主训练仍采用三项密度的直接形式。

In [ ]:
with torch.no_grad():
    m, log_s = posterior_params(Y[:1])
    q = torch.distributions.Normal(m[0], log_s[0].exp())
    prior = torch.distributions.Normal(0., 1.)
    analytic_kl = 0.5 * (m[0]**2 + log_s[0].exp()**2 - 1 - 2*log_s[0])
    total = 0.0
    M = 10000
    for j in range(M):
        x = q.sample()
        total += q.log_prob(x) - prior.log_prob(x)
    print('analytic KL:', analytic_kl.item())
    print('sampled KL:', (total / M).item())